# NB4 follow-up — do code-grounded, rewritten training targets close the judge-score gap?

Standalone notebook, staged so nothing expensive runs before cheaper checks pass:

1. **Step A (no GPU, ~5 min):** rewrite a 150-row sample of the existing 400-row LoRA
   training targets with a code-grounded GPT-4o pass, then score the ORIGINAL and
   REWRITTEN docstrings against the actual function code with the same code-aware judge
   built in the main notebook's Step 6c. Paired bootstrap on the quality-score
   difference - a real, honest test of whether the rewrite is actually better, before
   any training happens.
2. **Step B (no GPU, ~1 min):** a groundedness spot-check on ~12 rewritten rows - does
   the rewrite claim anything the code doesn't actually support (hallucination risk).
3. **Gate:** if Step A doesn't show a real improvement, the notebook stops here -
   training on unproven "improved" targets isn't worth the GPU time.
4. **Steps C-E (GPU, ~30-40 min if the gate passes):** a controlled pilot - two LoRA
   runs at the SAME size and hyperparameters as the original `LoRA r=8` baseline
   (400 rows, r=8, 2 epochs), one on the original targets and one on the rewritten
   targets, generated and judged (GPT + Gemini) on the same 70-row eval sample as
   every other run in this project, with bootstrap comparisons isolating the rewrite's
   effect specifically (rewritten vs. original, not just vs. base).

**Before running:** `OPENAI_API_KEY` and `GEMINI_API_KEY` need to be set in this
Colab's secrets, same as the main notebook. The next cell tries to restore NB1-3
outputs from your Drive backup automatically; if it can't, it rebuilds them from
scratch (a few extra minutes, nothing to do manually).

In [1]:
from pathlib import Path
import shutil, zipfile

try:
    from google.colab import drive
    drive.mount('/content/drive', force_remount=False)
    _drive_zip = Path('/content/drive/MyDrive/codesearernet_output.zip')
    if _drive_zip.exists():
        print(f"Found {_drive_zip} - extracting so this notebook can reuse NB1-3's outputs instead of rebuilding them...")
        with zipfile.ZipFile(_drive_zip) as zf:
            zf.extractall('.')
        _extracted_root = Path('codesearernet_output')
        if _extracted_root.exists():
            for _sub in _extracted_root.iterdir():
                _dest = Path(_sub.name)
                if _dest.exists():
                    shutil.rmtree(_dest)
                shutil.move(str(_sub), str(_dest))
            shutil.rmtree(_extracted_root, ignore_errors=True)
        print("Done - NB1-3 outputs restored from Drive; this notebook will load them from disk instead of rebuilding.")
    else:
        print(f"No backup zip found at {_drive_zip} - this notebook will rebuild NB1-3's outputs from scratch "
              "(a few extra minutes, handled automatically below).")
except Exception as e:
    print(f"Could not mount Drive / restore the backup ({e}) - continuing; NB1-3 outputs will be rebuilt from "
          "scratch below if not already present in this Colab session's local files.")


Mounted at /content/drive
Found /content/drive/MyDrive/codesearernet_output.zip - extracting so this notebook can reuse NB1-3's outputs instead of rebuilding them...
Done - NB1-3 outputs restored from Drive; this notebook will load them from disk instead of rebuilding.


## Setup

In [2]:
!pip install -q transformers accelerate peft sentencepiece pandas pyarrow datasets langdetect rouge-score matplotlib openai huggingface_hub sentence-transformers bitsandbytes google-genai
!pip install -q -U "torchao>=0.16.0"
# If this cell is being re-run in a session that already imported the old
# torchao (e.g. a previous attempt at this notebook), the upgrade above won't
# take effect until the Python process restarts - Runtime -> Restart session,
# then run all cells again from the top, is the reliable fix if Step 2.3
# still raises the torchao ImportError after this cell.


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 981.5/981.5 kB 32.5 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.1/43.1 MB 62.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.4/3.4 MB 79.2 MB/s eta 0:00:00


In [3]:
import os
import gc
import json as _json
import re
import ast
import textwrap
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import matplotlib.pyplot as plt

pd.set_option("display.max_colwidth", 200)

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device: {device}")
if device == "cpu":
    print("WARNING: no GPU detected. LoRA training will be extremely slow or "
          "may run out of memory on the 3B model. Switch to a GPU runtime "
          "(Runtime -> Change runtime type) before continuing.")


Device: cuda


## Data loading

Verbatim from the main notebook's Step 0.1/0.2 - same constants, same seeds, same
filters, loaded from disk if present (see the Drive-restore cell above).

In [4]:
PIPELINE_OUTPUT_DIR = Path("codesearchnet_pipeline_output")        # NB2's output dir
NB3_OUTPUT_DIR = Path("codesearchnet_lora_prep_output")            # NB3's output dir
NB4_OUTPUT_DIR = Path("codesearchnet_lora_train_output")           # this notebook's own output dir
PIPELINE_OUTPUT_DIR.mkdir(exist_ok=True)
NB3_OUTPUT_DIR.mkdir(exist_ok=True)
NB4_OUTPUT_DIR.mkdir(exist_ok=True)

repo_col = "repository_name"
doc_col = "func_documentation_string"
code_col = "func_code_string"
path_col = "func_path_in_repository"

# NB1/NB2/NB3's own parameters, reproduced verbatim.
WORKING_SAMPLE_SIZE = 30000
RANDOM_SEED = 42
MIN_DOC_WORDS = 3
EMBEDDING_MODEL_NAME = "sentence-transformers/all-MiniLM-L6-v2"   # NB1/NB2's embedding model, used only for its tokenizer
TOKEN_BUDGET = 512
SPLIT_SEED = 123
TRAIN_FRAC, VAL_FRAC = 0.8, 0.1
LORA_SUBSAMPLE_SIZE = 400
MAX_PER_REPO = 8
SUBSAMPLE_SEED = 77

# Full fine-tuning updates 100% of the model's parameters rather than LoRA's
# ~0.12-0.24% (Step 2.1/2.2 below), so it needs more data to generalize from
# and is more exposed to any single repo's coding/doc conventions dominating
# the update - a larger, still repo-capped subsample, not the 400-row LoRA
# one, is used for it (Step 0.1b below and Step 2.2's training function).
FULL_FINETUNE_SUBSAMPLE_SIZE = 3000
FULL_FINETUNE_MAX_PER_REPO = 15

# Step 3.9's finding (a later run of this notebook): every trained run's
# generated docstrings collapsed to a fraction of the base model's average
# length (55.6 words -> 9-26 words), and Step 3.10's manual audit confirmed
# real content was lost, not just words - not a judge-calibration artifact.
# The leading explanation is that supervised fine-tuning correctly teaches
# the model to match the TRAINING SUBSAMPLE's own docstring length/detail,
# and that subsample (an unfiltered repo-capped sample of real GitHub
# docstrings) is itself much terser on average than what the base model
# writes unprompted. MIN_DETAILED_DOC_WORDS filters the source pool for a
# second LoRA subsample (Step 0.1's lora_train_df_detailed) to only rows
# whose docstring is at least this long, to test that explanation directly -
# same size (LORA_SUBSAMPLE_SIZE) and repo cap (MAX_PER_REPO) as the original
# lora_train_df, so training-data length/detail is the ONLY thing that
# differs from the existing "LoRA r=8" variant.
MIN_DETAILED_DOC_WORDS = 15  # ~5x NB1/NB2's original MIN_DOC_WORDS=3 filter; a deliberately conservative "more than a one-liner" bar, not an attempt to match the base model's own ~55-word average

# The detail-filtered run above (lora_train_df_detailed) narrowed the
# judge-score decline but did NOT close it - a real, bootstrap-confirmed
# decline remained even after filtering training data down to "detailed"
# docstrings only. Two follow-up questions that filtering alone can't
# answer, tested with two more runs at 3x the row count (1,200 rows instead
# of 400, same 2% per-repo cap ratio so repo balance stays comparable):
# is 400 training rows itself just too few to generalize well regardless of
# content (LARGE_LORA_SUBSAMPLE_SIZE, unfiltered), and does combining MORE
# data with the length filter close the gap further than either change did
# alone (the same size, but drawn from the detailed-only pool)?
LARGE_LORA_SUBSAMPLE_SIZE = 1200
LARGE_LORA_MAX_PER_REPO = 24  # same 2% ratio as LORA_SUBSAMPLE_SIZE=400 / MAX_PER_REPO=8
LARGE_LORA_SUBSAMPLE_SEED = 88  # distinct from SUBSAMPLE_SEED so this isn't just a superset draw of lora_train_df

# The "3x data" run above answered both follow-up questions cleanly: more
# UNFILTERED data alone made the judge-score decline WORSE (and introduced a
# real overfitting signal), while combining more data with the length filter
# produced the best-behaved run of the first eight - the leading LoRA
# candidate overall, but its judge-score decline was still real and
# bootstrap-confirmed, not fully closed. MIN_DETAILED_DOC_WORDS_STRICT pushes
# the length bar further (double the original filter) to test whether MORE
# detail keeps helping monotonically, or whether the sixth/eighth runs'
# partial recovery has already plateaued.
MIN_DETAILED_DOC_WORDS_STRICT = 30  # 2x MIN_DETAILED_DOC_WORDS


In [5]:
def strip_docstring(code_text):
    '''Removes a function's docstring from its source text (NB2 Step 4a /
    NB3 Step 0.1). Code shown to the model for the code->doc task must not
    already contain its own target answer verbatim - a real LoRA run
    documented in NB2/NB3 confirmed a model will learn to copy it back out
    word-for-word within ~50 training steps otherwise. Uses `ast` to find the
    docstring's exact line span; returns the text unchanged if it doesn't
    parse (rare corpus edge cases - safer to under-strip than to corrupt the
    code shown to the model).'''
    original_lines = str(code_text).splitlines(keepends=True)
    try:
        with warnings.catch_warnings():
            warnings.simplefilter("ignore")  # non-raw backslash escapes in real GitHub code -> noisy SyntaxWarning, unrelated to this function's job
            tree = ast.parse(textwrap.dedent(str(code_text)))
    except SyntaxError:
        return code_text
    for node in ast.walk(tree):
        if isinstance(node, (ast.FunctionDef, ast.AsyncFunctionDef)):
            if (node.body and isinstance(node.body[0], ast.Expr)
                    and isinstance(getattr(node.body[0], "value", None), ast.Constant)
                    and isinstance(node.body[0].value.value, str)):
                doc_node = node.body[0]
                start, end = doc_node.lineno - 1, doc_node.end_lineno
                new_lines = original_lines[:start] + original_lines[end:]
                return "".join(new_lines) if len(new_lines) < len(original_lines) else code_text
    return code_text


def _build_split_dataframes():
    '''Reproduces NB2's Steps 0-3, 5, 10 (load dataset, 3 quality filters,
    512-token budget filter, repo-grouped split, save) - the same fallback
    logic NB3 used. Skips NB2's Steps 6-9 (embeddings + Chroma), which this
    LoRA notebook doesn't need.'''
    from datasets import load_dataset
    from transformers import AutoTokenizer
    from langdetect import detect, DetectorFactory, LangDetectException

    print("No NB2 output found - reproducing NB1/NB2's data pipeline inline "
          "(same dataset, filters, thresholds and seeds). This downloads the "
          "dataset and runs the full filter pipeline - a few minutes.")

    try:
        ds = load_dataset("claudios/code_search_net", "python")
    except Exception as e:
        print("Failed to load claudios/code_search_net, falling back to the official version...")
        print(e)
        ds = load_dataset("code-search-net/code_search_net", "python", trust_remote_code=True)

    raw_sample = ds["train"].shuffle(seed=RANDOM_SEED).select(
        range(min(WORKING_SAMPLE_SIZE, len(ds["train"])))
    )
    df = raw_sample.to_pandas()
    local_code_col = code_col if code_col in df.columns else "whole_func_string"

    df["doc_len_words"] = df[doc_col].astype(str).str.split().str.len()
    df = df[df["doc_len_words"] >= MIN_DOC_WORDS].copy()
    df = df[~df[local_code_col].duplicated(keep="first")].copy()

    DetectorFactory.seed = 0

    def safe_detect(text):
        text = str(text).strip()
        if len(text) < 3:
            return "unknown"
        try:
            return detect(text)
        except LangDetectException:
            return "unknown"

    print("Running language detection (slowest filter step)...")
    df["doc_lang"] = df[doc_col].apply(safe_detect)
    df = df[df["doc_lang"] == "en"].copy()

    tokenizer = AutoTokenizer.from_pretrained(EMBEDDING_MODEL_NAME)
    df["chunk_text"] = df[doc_col].astype(str) + "\n\n" + df[local_code_col].astype(str)
    df["chunk_len_tokens"] = df["chunk_text"].apply(lambda x: len(tokenizer.encode(x, truncation=False)))
    df = df[df["chunk_len_tokens"] <= TOKEN_BUDGET].copy()

    df = df.reset_index(drop=True)
    df["chunk_id"] = df.index.astype(str)
    df["func_code_no_doc"] = df[local_code_col].apply(strip_docstring)

    unique_repos = df[repo_col].unique()
    rng = np.random.default_rng(SPLIT_SEED)
    rng.shuffle(unique_repos)
    n_repos = len(unique_repos)
    n_train_repos = int(n_repos * TRAIN_FRAC)
    n_val_repos = int(n_repos * VAL_FRAC)
    train_repos = set(unique_repos[:n_train_repos])
    val_repos = set(unique_repos[n_train_repos:n_train_repos + n_val_repos])

    def assign_split(repo):
        if repo in train_repos:
            return "train"
        elif repo in val_repos:
            return "val"
        return "test"

    df["split"] = df[repo_col].apply(assign_split)

    out = {}
    for split_name in ["train", "val", "test"]:
        split_df = df[df["split"] == split_name].reset_index(drop=True)
        out_path = PIPELINE_OUTPUT_DIR / f"{split_name}_df.parquet"
        split_df.to_parquet(out_path, index=False)
        out[split_name] = split_df
        print(f"Saved {len(split_df):,} rows -> {out_path}")

    return out["train"], out["val"], out["test"]


def _build_repo_balanced_subsample(train_df_in, target_size, max_per_repo, seed, out_path, out_label):
    '''Repo-capped subsample builder: caps every repo at max_per_repo rows
    first, then fills up to target_size - the same two-stage approach NB3
    Step 0.4 used for the 400-row LoRA subsample, generalized here so it can
    also build the larger subsample Step 2.2's full-fine-tune variant trains
    on. Capping first (rather than a single random sample of target_size)
    is what actually bounds any one repo's share of the result; a plain
    random sample would let a handful of large repos dominate proportionally
    to their size in train_df, exactly what this is meant to avoid.'''
    capped_parts = [
        g.sample(n=min(len(g), max_per_repo), random_state=seed)
        for _, g in train_df_in.groupby(repo_col)
    ]
    capped = pd.concat(capped_parts) if capped_parts else pd.DataFrame(columns=train_df_in.columns)

    if len(capped) >= target_size:
        out_df = capped.sample(n=target_size, random_state=seed).reset_index(drop=True)
    else:
        remaining = target_size - len(capped)
        fill_pool = train_df_in.drop(index=capped.index)
        fill_rows = fill_pool.sample(n=min(remaining, len(fill_pool)), random_state=seed)
        out_df = pd.concat([capped, fill_rows]).sample(frac=1, random_state=seed).reset_index(drop=True)

    out_df.to_parquet(out_path, index=False)
    print(f"Saved {len(out_df)} rows ({out_label}) -> {out_path}")
    return out_df


def _build_lora_subsample(train_df_in):
    '''Reproduces NB3 Step 0.4's balanced LoRA training subsample.'''
    return _build_repo_balanced_subsample(
        train_df_in, LORA_SUBSAMPLE_SIZE, MAX_PER_REPO, SUBSAMPLE_SEED,
        NB3_OUTPUT_DIR / "lora_train_subsample.parquet", "LoRA subsample",
    )


def _build_full_finetune_subsample(train_df_in):
    '''Builds the larger, still repo-capped subsample the full-fine-tune
    variant (Step 2.1/2.2) trains on - see the constants cell above for why
    this is bigger than the LoRA subsample and why it's still capped per
    repo rather than a plain random sample of the same size.'''
    return _build_repo_balanced_subsample(
        train_df_in, FULL_FINETUNE_SUBSAMPLE_SIZE, FULL_FINETUNE_MAX_PER_REPO, SUBSAMPLE_SEED,
        NB4_OUTPUT_DIR / "full_finetune_train_subsample.parquet", "full fine-tune subsample",
    )


def _build_detailed_lora_subsample(train_df_in):
    '''Builds the "detail-filtered" LoRA subsample used to test whether
    Step 3.9's generated-length collapse traces back to the training
    subsample's own docstring length - same size (LORA_SUBSAMPLE_SIZE) and
    repo cap (MAX_PER_REPO) as lora_train_df, so training-data length/detail
    is the only thing this isolates. Filters the SOURCE pool to
    MIN_DETAILED_DOC_WORDS+ before capping/sampling (not after), so every
    row the result can possibly contain already meets the bar - the
    fallback backfill in _build_repo_balanced_subsample then also only ever
    draws from this same filtered pool, never from the unfiltered rest of
    train_df.'''
    detailed_pool = train_df_in[
        train_df_in[doc_col].astype(str).str.split().str.len() >= MIN_DETAILED_DOC_WORDS
    ].copy()
    print(f"Detail filter: {len(detailed_pool):,} / {len(train_df_in):,} train_df rows have a docstring "
          f"of at least {MIN_DETAILED_DOC_WORDS} words.")
    result = _build_repo_balanced_subsample(
        detailed_pool, LORA_SUBSAMPLE_SIZE, MAX_PER_REPO, SUBSAMPLE_SEED,
        NB4_OUTPUT_DIR / "lora_train_subsample_detailed.parquet", "detail-filtered LoRA subsample",
    )
    if len(result) < LORA_SUBSAMPLE_SIZE:
        print(f"WARNING: only {len(result)}/{LORA_SUBSAMPLE_SIZE} rows available after the detail filter and "
              f"repo cap - the detail-filtered variant will train on fewer rows than lora_train_df, which is "
              f"itself a confound worth noting if this variant's results are read later.")
    return result


def _build_large_lora_subsample(train_df_in):
    '''Builds the "3x data, unfiltered" LoRA subsample - same sampling logic
    as _build_lora_subsample, just LARGE_LORA_SUBSAMPLE_SIZE rows instead of
    LORA_SUBSAMPLE_SIZE, at the same per-repo cap ratio. Tests whether 400
    rows is itself too few to generalize well, independent of the detail
    filter - this table is UNFILTERED (drawn straight from train_df, no
    docstring-length bar) so it isolates sample size as its own variable.'''
    return _build_repo_balanced_subsample(
        train_df_in, LARGE_LORA_SUBSAMPLE_SIZE, LARGE_LORA_MAX_PER_REPO, LARGE_LORA_SUBSAMPLE_SEED,
        NB4_OUTPUT_DIR / "lora_train_subsample_large.parquet", "3x-data LoRA subsample (unfiltered)",
    )


def _build_large_detailed_lora_subsample(train_df_in):
    '''Builds the "3x data, detail-filtered" LoRA subsample - combines both
    follow-up questions from the detail-filtered run's result (a real but
    incomplete recovery): same MIN_DETAILED_DOC_WORDS+ filter as
    lora_train_df_detailed, but LARGE_LORA_SUBSAMPLE_SIZE rows instead of
    LORA_SUBSAMPLE_SIZE. If this run closes the judge-score gap further than
    either the detail filter alone or (if it helps at all) more unfiltered
    data alone, that's evidence the two effects compound rather than being
    redundant explanations of the same thing.'''
    detailed_pool = train_df_in[
        train_df_in[doc_col].astype(str).str.split().str.len() >= MIN_DETAILED_DOC_WORDS
    ].copy()
    result = _build_repo_balanced_subsample(
        detailed_pool, LARGE_LORA_SUBSAMPLE_SIZE, LARGE_LORA_MAX_PER_REPO, LARGE_LORA_SUBSAMPLE_SEED,
        NB4_OUTPUT_DIR / "lora_train_subsample_detailed_large.parquet", "3x-data LoRA subsample (detail-filtered)",
    )
    if len(result) < LARGE_LORA_SUBSAMPLE_SIZE:
        print(f"WARNING: only {len(result)}/{LARGE_LORA_SUBSAMPLE_SIZE} rows available after the detail filter and "
              f"repo cap - the 3x detail-filtered variant will train on fewer rows than intended, which is "
              f"itself a confound worth noting if this variant's results are read later.")
    return result


def _build_large_strict_detailed_lora_subsample(train_df_in):
    '''Builds the "3x data, STRICTER detail filter" LoRA subsample - same
    scale (LARGE_LORA_SUBSAMPLE_SIZE rows, LARGE_LORA_MAX_PER_REPO/repo) as
    lora_train_df_detailed_large, but filtered to MIN_DETAILED_DOC_WORDS_STRICT
    (double the original bar) instead of MIN_DETAILED_DOC_WORDS. Isolates
    "does pushing the detail filter further keep helping" as its own
    variable, holding data quantity fixed at the 3x scale that already
    produced the best-behaved run so far.'''
    strict_pool = train_df_in[
        train_df_in[doc_col].astype(str).str.split().str.len() >= MIN_DETAILED_DOC_WORDS_STRICT
    ].copy()
    print(f"Strict detail filter: {len(strict_pool):,} / {len(train_df_in):,} train_df rows have a docstring "
          f"of at least {MIN_DETAILED_DOC_WORDS_STRICT} words.")
    result = _build_repo_balanced_subsample(
        strict_pool, LARGE_LORA_SUBSAMPLE_SIZE, LARGE_LORA_MAX_PER_REPO, LARGE_LORA_SUBSAMPLE_SEED,
        NB4_OUTPUT_DIR / "lora_train_subsample_detailed_strict_large.parquet", "3x-data LoRA subsample (stricter detail filter)",
    )
    if len(result) < LARGE_LORA_SUBSAMPLE_SIZE:
        print(f"WARNING: only {len(result)}/{LARGE_LORA_SUBSAMPLE_SIZE} rows available after the stricter detail "
              f"filter and repo cap - this variant will train on fewer rows than intended, which is itself a "
              f"confound worth noting if its results are read later.")
    return result


def repo_concentration_report(df, label):
    '''Empirical repo-bias check, the same kind of evidence-over-assumption
    reporting NB3 Step 0.4 used ("largest single repo 0.5%, top-5 repos
    2.5%") - rather than asserting a table is repo-balanced, print exactly
    how concentrated it actually is.'''
    if len(df) == 0:
        return {"table": label, "n_rows": 0, "n_repos": 0, "top1_repo_pct": float("nan"), "top5_repo_pct": float("nan")}
    counts = df[repo_col].value_counts(normalize=True) * 100
    return {
        "table": label,
        "n_rows": len(df),
        "n_repos": df[repo_col].nunique(),
        "top1_repo_pct": round(counts.iloc[0], 2),
        "top5_repo_pct": round(counts.head(5).sum(), 2),
    }


In [6]:
# ---- Load or rebuild train_df / val_df / test_df ----
_train_path = PIPELINE_OUTPUT_DIR / "train_df.parquet"
_val_path = PIPELINE_OUTPUT_DIR / "val_df.parquet"
_test_path = PIPELINE_OUTPUT_DIR / "test_df.parquet"

if _train_path.exists() and _val_path.exists() and _test_path.exists():
    train_df = pd.read_parquet(_train_path)
    val_df = pd.read_parquet(_val_path)
    # test_df is loaded here (not skipped) because the fallback split-rebuild path
    # below produces train/val/test together and test's rows must exist on disk for
    # the split to be reproducible later - but it is NOT inspected anywhere in this
    # notebook (no row count, no content, no stats printed or plotted) and is never
    # read from again after this cell. It stays untouched until Step 6 has actually
    # picked a final run - see Step 0.1b's note for why.
    test_df = pd.read_parquet(_test_path)
    print(f"Loaded NB2 output from disk: train={len(train_df):,}  val={len(val_df):,}  (test_df loaded but not inspected)")
    if "func_code_no_doc" not in train_df.columns:
        print("train_df is missing 'func_code_no_doc' (older NB2 output) - computing it now...")
        train_df["func_code_no_doc"] = train_df[code_col].apply(strip_docstring)
    if "func_code_no_doc" not in val_df.columns:
        val_df["func_code_no_doc"] = val_df[code_col].apply(strip_docstring)
else:
    train_df, val_df, test_df = _build_split_dataframes()

# ---- Load or rebuild the balanced LoRA training subsample ----
_lora_subsample_path = NB3_OUTPUT_DIR / "lora_train_subsample.parquet"
if _lora_subsample_path.exists():
    lora_train_df = pd.read_parquet(_lora_subsample_path)
    print(f"Loaded NB3 LoRA subsample from disk: {len(lora_train_df):,} rows")
    if "func_code_no_doc" not in lora_train_df.columns:
        lora_train_df["func_code_no_doc"] = lora_train_df[code_col].apply(strip_docstring)
else:
    lora_train_df = _build_lora_subsample(train_df)

# ---- Load or rebuild the larger, repo-balanced full-fine-tune subsample ----
_full_ft_subsample_path = NB4_OUTPUT_DIR / "full_finetune_train_subsample.parquet"
if _full_ft_subsample_path.exists():
    full_finetune_train_df = pd.read_parquet(_full_ft_subsample_path)
    print(f"Loaded full fine-tune subsample from disk: {len(full_finetune_train_df):,} rows")
    if "func_code_no_doc" not in full_finetune_train_df.columns:
        full_finetune_train_df["func_code_no_doc"] = full_finetune_train_df[code_col].apply(strip_docstring)
else:
    full_finetune_train_df = _build_full_finetune_subsample(train_df)

# ---- Load or rebuild the detail-filtered LoRA subsample (tests Step 3.9's length-collapse finding) ----
_detailed_subsample_path = NB4_OUTPUT_DIR / "lora_train_subsample_detailed.parquet"
if _detailed_subsample_path.exists():
    lora_train_df_detailed = pd.read_parquet(_detailed_subsample_path)
    print(f"Loaded detail-filtered LoRA subsample from disk: {len(lora_train_df_detailed):,} rows")
    if "func_code_no_doc" not in lora_train_df_detailed.columns:
        lora_train_df_detailed["func_code_no_doc"] = lora_train_df_detailed[code_col].apply(strip_docstring)
else:
    lora_train_df_detailed = _build_detailed_lora_subsample(train_df)

# ---- Load or rebuild the two "3x data" follow-up subsamples (does more data,
# filtered or not, close the gap the detail-filtered run only narrowed?) ----
_large_subsample_path = NB4_OUTPUT_DIR / "lora_train_subsample_large.parquet"
if _large_subsample_path.exists():
    lora_train_df_large = pd.read_parquet(_large_subsample_path)
    print(f"Loaded 3x-data LoRA subsample (unfiltered) from disk: {len(lora_train_df_large):,} rows")
    if "func_code_no_doc" not in lora_train_df_large.columns:
        lora_train_df_large["func_code_no_doc"] = lora_train_df_large[code_col].apply(strip_docstring)
else:
    lora_train_df_large = _build_large_lora_subsample(train_df)

_large_detailed_subsample_path = NB4_OUTPUT_DIR / "lora_train_subsample_detailed_large.parquet"
if _large_detailed_subsample_path.exists():
    lora_train_df_detailed_large = pd.read_parquet(_large_detailed_subsample_path)
    print(f"Loaded 3x-data LoRA subsample (detail-filtered) from disk: {len(lora_train_df_detailed_large):,} rows")
    if "func_code_no_doc" not in lora_train_df_detailed_large.columns:
        lora_train_df_detailed_large["func_code_no_doc"] = lora_train_df_detailed_large[code_col].apply(strip_docstring)
else:
    lora_train_df_detailed_large = _build_large_detailed_lora_subsample(train_df)

# ---- Load or rebuild the ninth run's subsample (does pushing the detail filter further keep helping?) ----
_large_strict_subsample_path = NB4_OUTPUT_DIR / "lora_train_subsample_detailed_strict_large.parquet"
if _large_strict_subsample_path.exists():
    lora_train_df_detailed_strict_large = pd.read_parquet(_large_strict_subsample_path)
    print(f"Loaded 3x-data LoRA subsample (stricter detail filter) from disk: {len(lora_train_df_detailed_strict_large):,} rows")
    if "func_code_no_doc" not in lora_train_df_detailed_strict_large.columns:
        lora_train_df_detailed_strict_large["func_code_no_doc"] = lora_train_df_detailed_strict_large[code_col].apply(strip_docstring)
else:
    lora_train_df_detailed_strict_large = _build_large_strict_detailed_lora_subsample(train_df)

print(f"\nFinal working tables: train_df={len(train_df):,}  val_df={len(val_df):,}  "
      f"lora_train_df={len(lora_train_df):,}  full_finetune_train_df={len(full_finetune_train_df):,}  "
      f"lora_train_df_detailed={len(lora_train_df_detailed):,}  "
      f"lora_train_df_large={len(lora_train_df_large):,}  "
      f"lora_train_df_detailed_large={len(lora_train_df_detailed_large):,}  "
      f"lora_train_df_detailed_strict_large={len(lora_train_df_detailed_strict_large):,}  "
      f"(test_df is on disk but deliberately not summarized/printed here - see Step 0.1b)")


No NB2 output found - reproducing NB1/NB2's data pipeline inline (same dataset, filters, thresholds and seeds). This downloads the dataset and runs the full filter pipeline - a few minutes.


README.md:   0%|          | 0.00/13.6k [00:00<?, ?B/s]

python/train-00000-of-00003.parquet: reconstructing file:   0%|          |  0.00B /  130MB            

python/train-00000-of-00003.parquet: downloading bytes:           |  0.00B            

python/train-00001-of-00003.parquet: reconstructing file:   0%|          |  0.00B /  135MB            

...

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

[transformers] Token indices sequence length is longer than the specified maximum sequence length for this model (793 > 512). Running this sequence through the model will result in indexing errors


Saved 18,730 rows -> codesearchnet_pipeline_output/train_df.parquet
Saved 1,986 rows -> codesearchnet_pipeline_output/val_df.parquet
Saved 2,178 rows -> codesearchnet_pipeline_output/test_df.parquet
Saved 400 rows (LoRA subsample) -> codesearchnet_lora_prep_output/lora_train_subsample.parquet
Saved 3000 rows (full fine-tune subsample) -> codesearchnet_lora_train_output/full_finetune_train_subsample.parquet
Detail filter: 10,256 / 18,730 train_df rows have a docstring of at least 15 words.
Saved 400 rows (detail-filtered LoRA subsample) -> codesearchnet_lora_train_output/lora_train_subsample_detailed.parquet
Saved 1200 rows (3x-data LoRA subsample (unfiltered)) -> codesearchnet_lora_train_output/lora_train_subsample_large.parquet
Saved 1200 rows (3x-data LoRA subsample (detail-filtered)) -> codesearchnet_lora_train_output/lora_train_subsample_detailed_large.parquet
Strict detail filter: 5,447 / 18,730 train_df rows have a docstring of at least 30 words.
Saved 1200 rows (3x-data LoRA sub

In [7]:
TASK_DIRECTION_DECISION = "code_to_doc"   # NB3 Step 0.1: every model scored higher on code->doc than doc->code
LORA_TARGET_MODEL_LABEL = "Qwen2.5-Coder-3B"
LORA_TARGET_MODEL_ID = "Qwen/Qwen2.5-Coder-3B-Instruct"

CARRIED_OVER_RATIONALE = (
    "code_to_doc: every one of NB3's 8 candidate models scored noticeably higher on the "
    "GPT judge for code->doc than doc->code, matching the manual read (NB3 Step 0.1c). "
    "Qwen2.5-Coder-3B: not the raw composite leader (Llama-3.2-3B was, by a narrow margin) "
    "but chosen because the bootstrap confidence check (NB3 Step 0.1d/e, both n=20 and a "
    "widened n=40) showed Llama's lead was real on only 1 of 5 metrics, while Qwen2.5-Coder-3B "
    "is ungated and code-specialized. See NB3's DECISION_RATIONALE for the full comparison."
)

print(f"Task direction : {TASK_DIRECTION_DECISION}")
print(f"LoRA target     : {LORA_TARGET_MODEL_LABEL}  ({LORA_TARGET_MODEL_ID})")
print(f"Rationale (from NB3): {CARRIED_OVER_RATIONALE}")


Task direction : code_to_doc
LoRA target     : Qwen2.5-Coder-3B  (Qwen/Qwen2.5-Coder-3B-Instruct)
Rationale (from NB3): code_to_doc: every one of NB3's 8 candidate models scored noticeably higher on the GPT judge for code->doc than doc->code, matching the manual read (NB3 Step 0.1c). Qwen2.5-Coder-3B: not the raw composite leader (Llama-3.2-3B was, by a narrow margin) but chosen because the bootstrap confidence check (NB3 Step 0.1d/e, both n=20 and a widened n=40) showed Llama's lead was real on only 1 of 5 metrics, while Qwen2.5-Coder-3B is ungated and code-specialized. See NB3's DECISION_RATIONALE for the full comparison.


## Training-pair construction (verbatim from the main notebook's Step 1)

In [8]:
CODE_TO_DOC_PROMPT = (
    "Write a concise, accurate docstring for the following Python function. "
    "Only output the docstring text - no code fences, no repetition of the function signature.\n\n"
    "```python\n{code}\n```"
)


def build_chat_messages(code_no_doc, true_doc=None):
    '''Builds the chat message list for one example. `true_doc=None` gives
    the user-turn only (for generation at eval time); a real string appends
    the assistant turn (for supervised training).'''
    messages = [{"role": "user", "content": CODE_TO_DOC_PROMPT.format(code=code_no_doc)}]
    if true_doc is not None:
        messages.append({"role": "assistant", "content": str(true_doc).strip()})
    return messages


In [9]:
from transformers import AutoTokenizer

print(f"Loading tokenizer for {LORA_TARGET_MODEL_LABEL}...")
tokenizer = AutoTokenizer.from_pretrained(LORA_TARGET_MODEL_ID)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

MAX_SEQ_LEN = 768  # generous vs. NB3 Step 0.3's measured median (~164 tokens) and 512-budget rows


def normalize_token_ids(x):
    '''`tokenizer.apply_chat_template(tokenize=True, ...)` does not return a
    consistent shape across transformers/tokenizers versions: depending on
    what's installed in this Colab session it can hand back a flat
    List[int] (the documented default), a BatchEncoding (dict-like, with the
    ids under "input_ids"), a raw tokenizers.Encoding object (with the ids
    under .ids), or a batched List[List[int]] for what is really a single
    conversation. Feeding any of the non-flat-list shapes straight into
    list slicing / tokenizer.decode() fails with a confusing low-level error
    ("'tokenizers.Encoding' object cannot be interpreted as an integer") -
    this is the exact failure NB3's own generate_response() already had to
    guard against for a related apply_chat_template quirk (its
    isinstance(..., BatchEncoding) check). This normalizes every shape down
    to a single flat list[int] up front, so the rest of the pipeline never
    has to special-case it again.'''
    if hasattr(x, "keys") and "input_ids" in x:      # BatchEncoding / dict-like
        x = x["input_ids"]
    if hasattr(x, "ids") and not isinstance(x, (list, tuple)):  # raw tokenizers.Encoding
        return list(x.ids)
    if isinstance(x, list) and len(x) > 0 and isinstance(x[0], (list, tuple)):  # batch-of-one
        x = x[0]
    return list(x)


def encode_supervised_example(code_no_doc, true_doc, max_len=MAX_SEQ_LEN):
    prompt_ids = normalize_token_ids(tokenizer.apply_chat_template(
        build_chat_messages(code_no_doc), add_generation_prompt=True, tokenize=True,
    ))
    full_ids = normalize_token_ids(tokenizer.apply_chat_template(
        build_chat_messages(code_no_doc, true_doc), add_generation_prompt=False, tokenize=True,
    ))
    if len(full_ids) < len(prompt_ids):
        # Defensive: a chat template edge case shouldn't happen, but silently
        # training on a negative-length response would corrupt the batch.
        return None

    full_ids = full_ids[:max_len]
    labels = list(full_ids)
    prompt_len = min(len(prompt_ids), len(full_ids))
    for i in range(prompt_len):
        labels[i] = -100
    return {"input_ids": full_ids, "labels": labels}


Loading tokenizer for Qwen2.5-Coder-3B...


config.json:   0%|          | 0.00/661 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/7.30k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/7.03M [00:00<?, ?B/s]

In [10]:
class SupervisedDocGenDataset(torch.utils.data.Dataset):
    '''Wraps a DataFrame of (func_code_no_doc, true_doc) pairs as tokenized,
    response-masked training examples. Rows that fail to encode (rare - see
    encode_supervised_example's defensive check) are dropped, not silently
    zero-padded into the batch.'''

    def __init__(self, df, code_col_no_doc="func_code_no_doc", doc_col_name=doc_col):
        self.examples = []
        n_dropped = 0
        for _, row in df.iterrows():
            ex = encode_supervised_example(row[code_col_no_doc], row[doc_col_name])
            if ex is None:
                n_dropped += 1
                continue
            self.examples.append(ex)
        if n_dropped:
            print(f"  Dropped {n_dropped} row(s) that failed to encode.")

    def __len__(self):
        return len(self.examples)

    def __getitem__(self, idx):
        return self.examples[idx]


def collate_supervised(batch):
    '''Right-pads input_ids/labels to the batch's longest sequence.
    Padded label positions get -100 (ignored by the loss), padded input_ids
    get the pad token with a matching 0 in attention_mask.'''
    max_len = max(len(ex["input_ids"]) for ex in batch)
    input_ids, attention_mask, labels = [], [], []
    for ex in batch:
        pad_n = max_len - len(ex["input_ids"])
        input_ids.append(ex["input_ids"] + [tokenizer.pad_token_id] * pad_n)
        attention_mask.append([1] * len(ex["input_ids"]) + [0] * pad_n)
        labels.append(ex["labels"] + [-100] * pad_n)
    return {
        "input_ids": torch.tensor(input_ids, dtype=torch.long),
        "attention_mask": torch.tensor(attention_mask, dtype=torch.long),
        "labels": torch.tensor(labels, dtype=torch.long),
    }


## LoRA training function

Same hyperparameter defaults and the same `train_lora_variant()` function as the main
notebook's Step 2.1/2.2, copied verbatim. The pilot runs below use ONLY the defaults
here (`r=8`, `lora_alpha=16`, 2 epochs) - the exact same recipe as the original
`LoRA r=8` baseline - so the only thing that differs between the two pilot runs is the
training targets themselves.

In [11]:
from peft import LoraConfig, get_peft_model, PeftModel
from transformers import AutoModelForCausalLM, TrainingArguments, Trainer, TrainerCallback

ATTENTION_TARGET_MODULES = ["q_proj", "k_proj", "v_proj", "o_proj"]
LORA_DROPOUT = 0.05
LEARNING_RATE = 1.5e-4
NUM_TRAIN_EPOCHS = 2
PER_DEVICE_TRAIN_BATCH_SIZE = 2
GRADIENT_ACCUMULATION_STEPS = 8
FULL_FINETUNE_LEARNING_RATE = 2e-5   # unused unless train_full_finetune_variant() is called below - kept for consistency with the source cell
FULL_FINETUNE_NUM_TRAIN_EPOCHS = 2

LOSS_CURVE_EVAL_SIZE = 40
LOSS_CURVE_EVAL_SEED = 501
loss_curve_eval_df = val_df.sample(n=min(LOSS_CURVE_EVAL_SIZE, len(val_df)), random_state=LOSS_CURVE_EVAL_SEED)
print(f"Loss-curve eval slice: {len(loss_curve_eval_df)} rows from val_df (seed={LOSS_CURVE_EVAL_SEED})")


Loss-curve eval slice: 40 rows from val_df (seed=501)


In [12]:
class LossHistoryCallback(TrainerCallback):
    '''Collects (step, loss, grad_norm) / (step, eval_loss, eval_token_accuracy)
    pairs from Trainer's log stream, so the overfitting-diagnostic charts in
    Step 2.4-2.6 are built from the actual training run rather than
    re-parsing Trainer's console output. grad_norm and eval_token_accuracy are
    read defensively (`.get`) since grad_norm's presence depends on the
    installed transformers version, and eval_token_accuracy only exists once
    compute_metrics (defined below) is wired into the Trainer.'''

    def __init__(self):
        self.train_log = []
        self.eval_log = []

    def on_log(self, args, state, control, logs=None, **kwargs):
        if logs is None:
            return
        if "loss" in logs:
            self.train_log.append({
                "step": state.global_step,
                "loss": logs["loss"],
                "grad_norm": logs.get("grad_norm"),
            })
        if "eval_loss" in logs:
            self.eval_log.append({
                "step": state.global_step,
                "eval_loss": logs["eval_loss"],
                "eval_token_accuracy": logs.get("eval_token_accuracy"),
            })


def preprocess_logits_for_metrics(logits, labels):
    '''Reduces each eval batch's (batch, seq_len, vocab) float logits down to
    (batch, seq_len) argmax token ids immediately after the forward pass,
    before Trainer accumulates predictions across the whole eval split.
    Without this, keeping full-vocabulary logits in memory for every row of a
    ~150K-vocabulary causal LM would exhaust Colab's GPU memory well before
    evaluation finishes - the same one-thing-at-a-time memory discipline used
    elsewhere in this notebook (Step 2.2's load-one-model-at-a-time pattern).'''
    if isinstance(logits, tuple):
        logits = logits[0]
    return logits.argmax(dim=-1)


def compute_metrics(eval_pred):
    '''Next-token accuracy on response tokens only (prompt tokens are labeled
    -100 by encode_supervised_example and excluded here too), using the same
    causal shift the loss itself uses internally: the prediction at position
    i is compared against the label at position i+1. This is the "did the
    model actually get better at predicting the held-out docstrings"
    complement to eval_loss - a model can lower its loss (grow more confident
    on already-correct tokens) without its top-1 accuracy moving at all, so
    the two are read together in Step 2.5, not as substitutes for each other.'''
    predictions, labels = eval_pred
    if isinstance(predictions, tuple):
        predictions = predictions[0]
    predictions = np.asarray(predictions)
    labels = np.asarray(labels)
    pred_shifted = predictions[:, :-1]
    label_shifted = labels[:, 1:]
    mask = label_shifted != -100
    if mask.sum() == 0:
        return {"token_accuracy": float("nan")}
    correct = (pred_shifted == label_shifted) & mask
    return {"token_accuracy": float(correct.sum()) / float(mask.sum())}


def train_lora_variant(variant, train_dataset, eval_dataset, output_dir):
    '''variant may additionally set "weight_decay" (default 0.0 - the
    Trainer/TrainingArguments default this notebook used everywhere until
    the overfitting-mitigation runs), "lora_dropout" (default LORA_DROPOUT),
    and "use_best_checkpoint" (default False - when True, the Trainer
    checkpoints periodically and reloads the checkpoint with the LOWEST
    eval_loss at the end, instead of whatever the final training step
    happens to land on; the three regularization knobs are independent, so
    each overfitting-mitigation run below changes exactly one relative to
    its baseline, same "change one knob" discipline as the LR/epoch
    follow-ups in Step 2.1).'''
    label, r, alpha = variant["label"], variant["r"], variant["lora_alpha"]
    lr = variant.get("learning_rate", LEARNING_RATE)
    epochs = variant.get("num_train_epochs", NUM_TRAIN_EPOCHS)
    weight_decay = variant.get("weight_decay", 0.0)
    lora_dropout = variant.get("lora_dropout", LORA_DROPOUT)
    use_best_checkpoint = variant.get("use_best_checkpoint", False)
    print(f"\n{'='*70}\nTraining {label} (lora_alpha={alpha}, lr={lr}, epochs={epochs}, "
          f"weight_decay={weight_decay}, lora_dropout={lora_dropout}, "
          f"use_best_checkpoint={use_best_checkpoint})\n{'='*70}")

    base_model = AutoModelForCausalLM.from_pretrained(
        LORA_TARGET_MODEL_ID,
        dtype=torch.bfloat16 if device == "cuda" else torch.float32,
        device_map=device,
    )
    lora_config = LoraConfig(
        r=r,
        lora_alpha=alpha,
        target_modules=ATTENTION_TARGET_MODULES,
        lora_dropout=lora_dropout,
        bias="none",
        task_type="CAUSAL_LM",
    )
    peft_model = get_peft_model(base_model, lora_config)
    peft_model.print_trainable_parameters()

    _ckpt_kwargs = (
        # Checkpoint every eval_steps and keep only the best-by-eval_loss one
        # on disk (save_total_limit=1) - load_best_model_at_end then reloads
        # it into `trainer.model` after training finishes, so what gets
        # saved as this run's adapter below is the least-overfit checkpoint
        # the Trainer actually saw, not whichever step training happened to
        # end on.
        dict(save_strategy="steps", save_steps=10, save_total_limit=1,
             load_best_model_at_end=True, metric_for_best_model="eval_loss", greater_is_better=False)
        if use_best_checkpoint else
        dict(save_strategy="no")  # adapters are saved explicitly below, not via checkpointing
    )
    training_args = TrainingArguments(
        output_dir=str(output_dir / "trainer_ckpts"),
        per_device_train_batch_size=PER_DEVICE_TRAIN_BATCH_SIZE,
        gradient_accumulation_steps=GRADIENT_ACCUMULATION_STEPS,
        num_train_epochs=epochs,
        learning_rate=lr,
        weight_decay=weight_decay,
        logging_steps=2,
        eval_strategy="steps",
        eval_steps=10,
        eval_accumulation_steps=4,  # offload eval predictions to CPU periodically - see preprocess_logits_for_metrics's note
        bf16=(device == "cuda"),
        report_to=[],
        remove_unused_columns=False,
        **_ckpt_kwargs,
    )

    loss_cb = LossHistoryCallback()
    trainer = Trainer(
        model=peft_model,
        args=training_args,
        train_dataset=train_dataset,
        eval_dataset=eval_dataset,
        data_collator=collate_supervised,
        callbacks=[loss_cb],
        compute_metrics=compute_metrics,
        preprocess_logits_for_metrics=preprocess_logits_for_metrics,
    )
    trainer.train()
    if use_best_checkpoint and trainer.state.best_model_checkpoint:
        print(f"Best checkpoint by eval_loss: {trainer.state.best_model_checkpoint} "
              f"(eval_loss={trainer.state.best_metric:.4f}) - reloaded into the model saved below.")

    adapter_path = output_dir / label.replace(" ", "_").replace("=", "")
    peft_model.save_pretrained(str(adapter_path))
    print(f"Saved adapter -> {adapter_path}")

    train_history = pd.DataFrame(loss_cb.train_log)
    eval_history = pd.DataFrame(loss_cb.eval_log)

    # Free the GPU before the next variant loads a fresh base model - same
    # discipline NB3 used across its 8 candidate models. Also clear out this
    # run's checkpoint directory when it used save_strategy="steps" - the
    # adapter itself is already saved above, so the raw Trainer checkpoint
    # (a full base-model-sized copy) is pure disk bloat once training ends.
    del trainer, peft_model, base_model
    gc.collect()
    if device == "cuda":
        torch.cuda.empty_cache()
    if use_best_checkpoint:
        import shutil
        shutil.rmtree(output_dir / "trainer_ckpts", ignore_errors=True)

    return {"label": label, "kind": "lora", "adapter_path": str(adapter_path),
            "train_history": train_history, "eval_history": eval_history}


def train_full_finetune_variant(label, train_dataset, eval_dataset, output_dir):
    '''Trains every parameter of the base model (no LoRA wrapper at all), on
    the larger full_finetune_train_df - see Step 2.1's markdown for why each
    setting here differs from train_lora_variant()'s. Wrapped in a try/except
    for CUDA out-of-memory specifically: this is the one run genuinely at
    risk of not fitting in whatever GPU Colab assigns, and a failure here
    should not take down the rest of the notebook - it returns None instead,
    and Step 2.3 skips adding it to training_runs when that happens.'''
    print(f"\n{'='*70}\nTraining {label} (full fine-tune, lr={FULL_FINETUNE_LEARNING_RATE}, "
          f"epochs={FULL_FINETUNE_NUM_TRAIN_EPOCHS})\n{'='*70}")

    model = None
    trainer = None
    try:
        model = AutoModelForCausalLM.from_pretrained(
            LORA_TARGET_MODEL_ID,
            dtype=torch.bfloat16 if device == "cuda" else torch.float32,
            device_map=device,
        )
        model.gradient_checkpointing_enable()
        n_trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
        n_total = sum(p.numel() for p in model.parameters())
        print(f"trainable params: {n_trainable:,} / {n_total:,} (100.00% - every parameter, by design)")

        training_args = TrainingArguments(
            output_dir=str(output_dir / "trainer_ckpts_full_ft"),
            per_device_train_batch_size=PER_DEVICE_TRAIN_BATCH_SIZE,
            gradient_accumulation_steps=GRADIENT_ACCUMULATION_STEPS,
            num_train_epochs=FULL_FINETUNE_NUM_TRAIN_EPOCHS,
            learning_rate=FULL_FINETUNE_LEARNING_RATE,
            optim="adamw_bnb_8bit",     # 8-bit optimizer state - full fp32 Adam state won't fit alongside a 3B model on one Colab GPU
            logging_steps=2,
            eval_strategy="steps",
            eval_steps=10,
            eval_accumulation_steps=4,
            save_strategy="no",
            bf16=(device == "cuda"),
            report_to=[],
            remove_unused_columns=False,
        )

        loss_cb = LossHistoryCallback()
        trainer = Trainer(
            model=model,
            args=training_args,
            train_dataset=train_dataset,
            eval_dataset=eval_dataset,
            data_collator=collate_supervised,
            callbacks=[loss_cb],
            compute_metrics=compute_metrics,
            preprocess_logits_for_metrics=preprocess_logits_for_metrics,
        )
        trainer.train()

        model_path = output_dir / label.replace(" ", "_").replace("(", "").replace(")", "")
        trainer.save_model(str(model_path))
        print(f"Saved full fine-tuned model -> {model_path}")

        train_history = pd.DataFrame(loss_cb.train_log)
        eval_history = pd.DataFrame(loss_cb.eval_log)
        result = {"label": label, "kind": "full", "model_path": str(model_path),
                  "train_history": train_history, "eval_history": eval_history}

    except torch.cuda.OutOfMemoryError as e:
        print(f"\nOUT OF MEMORY while training {label}: {e}")
        print("Skipping the full fine-tune run - this GPU doesn't have enough memory for it even with "
              "8-bit Adam and gradient checkpointing. Every other step in this notebook iterates over "
              "training_runs, so it will simply proceed with the LoRA variants alone.")
        result = None
    except RuntimeError as e:
        if "out of memory" in str(e).lower():
            print(f"\nOUT OF MEMORY (RuntimeError) while training {label}: {e}")
            print("Skipping the full fine-tune run - see the note above.")
            result = None
        else:
            raise

    del trainer, model
    gc.collect()
    if device == "cuda":
        torch.cuda.empty_cache()

    return result


In [13]:
loss_curve_eval_dataset = SupervisedDocGenDataset(loss_curve_eval_df)
print(f"Loss-curve eval examples: {len(loss_curve_eval_dataset)}")


Loss-curve eval examples: 40


## Evaluation infrastructure

`eval_sample`, `generate_response()`, `run_generation_pass()`, the ROUGE-L/semantic-
similarity functions, and the GPT/Gemini judge functions are the exact same functions
as the main notebook's Step 3.1-3.4b, copied verbatim (execution loops left out - this
notebook builds its own generation_df further down).

In [14]:
BEFORE_AFTER_EVAL_SIZE = 70   # NB3's own escalation path: n=20 -> widened n=40; a fresh, independent n=70 here
BEFORE_AFTER_EVAL_SEED = 913  # disjoint from LOSS_CURVE_EVAL_SEED (501) and NB3's seeds (11, 29)
MAX_NEW_TOKENS = 260          # matches NB3's (raised) code->doc budget - see NB3 Step 0.1 for why 150 was too tight

eval_sample = val_df.sample(n=min(BEFORE_AFTER_EVAL_SIZE, len(val_df)), random_state=BEFORE_AFTER_EVAL_SEED).reset_index(drop=True)
print(f"Before/after eval sample: {len(eval_sample)} rows from val_df (seed={BEFORE_AFTER_EVAL_SEED})")


def generate_response(tok, model, messages, max_new_tokens=MAX_NEW_TOKENS):
    '''Same deterministic single-turn generation helper as NB3's. Also uses
    the same apply_chat_template-shape defense as Step 1.2's
    normalize_token_ids(): with return_tensors="pt", this call is usually a
    plain tensor (NB3's own assumption), but on some transformers versions
    comes back as a BatchEncoding instead - unwrap it if so, rather than
    letting a version mismatch surface as a confusing .shape/.to() error
    mid-generation.'''
    processed = tok.apply_chat_template(messages, add_generation_prompt=True, return_tensors="pt")
    input_ids = processed["input_ids"] if hasattr(processed, "keys") else processed
    input_ids = input_ids.to(model.device)
    with torch.no_grad():
        output_ids = model.generate(
            input_ids=input_ids,
            max_new_tokens=max_new_tokens,
            do_sample=False,
            pad_token_id=tok.eos_token_id if tok.eos_token_id is not None else tok.pad_token_id,
        )
    new_tokens = output_ids[0][input_ids.shape[1]:]
    truncated = len(new_tokens) >= max_new_tokens
    return tok.decode(new_tokens, skip_special_tokens=True).strip(), truncated


Before/after eval sample: 70 rows from val_df (seed=913)


In [15]:
def run_generation_pass(run_label, model, tok, sample_df):
    rows = []
    for i, row in sample_df.iterrows():
        gen_text, truncated = generate_response(
            tok, model, build_chat_messages(row["func_code_no_doc"]),
        )
        rows.append({
            "sample_idx": i,
            "run": run_label,
            "repository": row[repo_col],
            "true_doc": row[doc_col],
            "generated_doc": gen_text,
            "truncated": truncated,
        })
    return pd.DataFrame(rows)


In [16]:
from rouge_score import rouge_scorer
from sentence_transformers import SentenceTransformer

scorer = rouge_scorer.RougeScorer(["rougeL"], use_stemmer=True)


def rouge_l(true_text, gen_text):
    if not str(true_text).strip() or not str(gen_text).strip():
        return 0.0
    return scorer.score(str(true_text), str(gen_text))["rougeL"].fmeasure


print(f"Loading {EMBEDDING_MODEL_NAME} for semantic-similarity scoring...")
_embed_model = SentenceTransformer(EMBEDDING_MODEL_NAME, device=device)


def semantic_cos_sim(true_text, gen_text):
    if not str(true_text).strip() or not str(gen_text).strip():
        return 0.0
    embs = _embed_model.encode([str(true_text), str(gen_text)], normalize_embeddings=True)
    return float(np.dot(embs[0], embs[1]))


Loading sentence-transformers/all-MiniLM-L6-v2 for semantic-similarity scoring...


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 90.9MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

In [17]:
def _get_openai_key():
    try:
        from google.colab import userdata
        return userdata.get("OPENAI_API_KEY")
    except Exception:
        return os.environ.get("OPENAI_API_KEY")


_openai_key = _get_openai_key()
JUDGE_PROMPT = (
    "You are judging the quality of an AI-generated Python docstring against the "
    "ground-truth human-written docstring for the same function.\n\n"
    "Ground-truth docstring:\n{true_doc}\n\n"
    "AI-generated docstring:\n{gen_doc}\n\n"
    "Score the AI-generated docstring's match to the ground truth's meaning and "
    "accuracy on a 1-5 scale (5 = equivalent in meaning and accuracy, 1 = wrong or "
    "unrelated). Reply with ONLY the integer score, nothing else."
)

if not _openai_key:
    print("No OPENAI_API_KEY found - the GPT judge, the rewrite step, and the quality "
          "check will all be skipped.")
else:
    from openai import OpenAI
    _client = OpenAI(api_key=_openai_key)

    def judge_score(true_doc, gen_doc):
        try:
            resp = _client.chat.completions.create(
                model="gpt-4o-mini",
                messages=[{"role": "user", "content": JUDGE_PROMPT.format(true_doc=true_doc, gen_doc=gen_doc)}],
                max_tokens=5,
                temperature=0,
            )
            return float(re.search(r"\d+", resp.choices[0].message.content).group())
        except Exception as e:
            print(f"  Judge call failed: {e}")
            return np.nan


In [18]:
def _get_gemini_key():
    try:
        from google.colab import userdata
        return userdata.get("GEMINI_API_KEY")
    except Exception:
        return os.environ.get("GEMINI_API_KEY")


_gemini_key = _get_gemini_key()

if not _gemini_key:
    print("No GEMINI_API_KEY found - the Gemini judge will be skipped for the pilot runs too.")
else:
    from google import genai as _genai
    _gemini_client = _genai.Client(api_key=_gemini_key)

    def gemini_judge_score(true_doc, gen_doc):
        try:
            resp = _gemini_client.models.generate_content(
                model="gemini-3.6-flash",
                contents=JUDGE_PROMPT.format(true_doc=true_doc, gen_doc=gen_doc),
            )
            return float(re.search(r"\d+", resp.text).group())
        except Exception as e:
            print(f"  Gemini judge call failed: {e}")
            return np.nan


In [19]:
def bootstrap_paired_diff(df, run_a, run_b, value_col, n_boot=5000, seed=0):
    pivot = df[df["run"].isin([run_a, run_b])].pivot(index="sample_idx", columns="run", values=value_col)
    pivot = pivot.dropna()
    if len(pivot) == 0 or run_a not in pivot.columns or run_b not in pivot.columns:
        return None
    a, b = pivot[run_a].to_numpy(), pivot[run_b].to_numpy()
    n = len(a)
    rng = np.random.default_rng(seed)
    idx = rng.integers(0, n, size=(n_boot, n))
    diffs = a[idx].mean(axis=1) - b[idx].mean(axis=1)
    point_est = a.mean() - b.mean()
    ci_low, ci_high = np.percentile(diffs, [2.5, 97.5])
    return point_est, ci_low, ci_high


## Step A: rewrite a validation sample, and check whether it's actually better

Rewrites 150 of `lora_train_df`'s 400 rows with a code-grounded GPT-4o pass (a
stronger model than the `gpt-4o-mini` used for judging elsewhere - the same
"stronger model for a more careful task" reasoning the main notebook's Step 3.10
already used for its second judge). Then scores BOTH the original and the rewritten
docstring against the actual function code with the same code-aware prompt the main
notebook's Step 6c validated, and runs a paired bootstrap on the difference - a real
test, not an assumption, of whether the rewrite is actually higher quality.

In [20]:
REWRITE_PROMPT = (
    "You are improving a Python function's docstring so it correctly and completely "
    "describes what the function does, based on its actual code.\n\n"
    "Function code:\n```python\n{code}\n```\n\n"
    "Existing docstring (may be incomplete, terse, or slightly imprecise):\n{orig_doc}\n\n"
    "Write a corrected, complete docstring for this function based on the code. Keep it "
    "concise (1-4 sentences, plus brief parameter/return notes only if the code has "
    "multiple non-obvious parameters or a non-trivial return value) - do not pad with "
    "filler, and do not state anything the code does not actually support. Reply with "
    "ONLY the docstring text, no code fences, no repetition of the function signature."
)

DOC_QUALITY_PROMPT = (
    "You are scoring whether a Python docstring correctly and completely describes "
    "what the function's code actually does.\n\nFunction code:\n```python\n{code}\n```\n\n"
    "Docstring:\n{doc}\n\nScore how correctly and completely this docstring describes "
    "the code on a 1-5 scale (5 = fully correct and complete, 1 = wrong, unrelated, or "
    "missing important behavior). Reply on one line in EXACTLY this format: "
    "SCORE: <int> | REASON: <one short sentence>."
)


def _parse_score_reason(text):
    score_match = re.search(r"SCORE:\s*(\d+)", text)
    reason_match = re.search(r"REASON:\s*(.*)", text, re.DOTALL)
    score = float(score_match.group(1)) if score_match else np.nan
    reason = reason_match.group(1).strip() if reason_match else text.strip()
    return score, reason


def rewrite_docstring(code_text, orig_doc):
    try:
        resp = _client.chat.completions.create(
            model="gpt-4o",
            messages=[{"role": "user", "content": REWRITE_PROMPT.format(code=code_text, orig_doc=orig_doc)}],
            max_tokens=200, temperature=0,
        )
        return resp.choices[0].message.content.strip()
    except Exception as e:
        print(f"  Rewrite call failed: {e}")
        return None


def doc_quality_score(code_text, doc):
    try:
        resp = _client.chat.completions.create(
            model="gpt-4o-mini",
            messages=[{"role": "user", "content": DOC_QUALITY_PROMPT.format(code=code_text, doc=doc)}],
            max_tokens=60, temperature=0,
        )
        return _parse_score_reason(resp.choices[0].message.content)
    except Exception as e:
        return np.nan, f"call failed: {e}"


if not _openai_key:
    raise AssertionError("No OPENAI_API_KEY - this entire notebook needs it (rewrite + both judges' quality checks).")

REWRITE_SAMPLE_SEED = 314
VALIDATION_SAMPLE_SIZE = 150
_rewrite_sample_df = lora_train_df.sample(n=min(VALIDATION_SAMPLE_SIZE, len(lora_train_df)), random_state=REWRITE_SAMPLE_SEED)
print(f"Rewriting and scoring {len(_rewrite_sample_df)} rows (of {len(lora_train_df)} in lora_train_df)...")

rewritten_cache = {}
quality_rows = []
for n, (idx, row) in enumerate(_rewrite_sample_df.iterrows()):
    code_text, orig_doc = row["func_code_no_doc"], row[doc_col]
    new_doc = rewrite_docstring(code_text, orig_doc)
    if new_doc is None:
        continue
    rewritten_cache[idx] = new_doc

    orig_score, _ = doc_quality_score(code_text, orig_doc)
    new_score, _ = doc_quality_score(code_text, new_doc)
    quality_rows.append({"sample_idx": idx, "run": "original", "quality_score": orig_score})
    quality_rows.append({"sample_idx": idx, "run": "rewritten", "quality_score": new_score})

    if (n + 1) % 30 == 0:
        print(f"  ...{n + 1}/{len(_rewrite_sample_df)} rewritten and scored")

quality_df = pd.DataFrame(quality_rows)
print(f"\nDone. {len(rewritten_cache)}/{len(_rewrite_sample_df)} rows successfully rewritten.")

_quality_summary = quality_df.groupby("run")["quality_score"].mean().round(3)
print("\nMean code-grounded quality score, original vs. rewritten:")
print(_quality_summary)

_quality_result = bootstrap_paired_diff(quality_df, "rewritten", "original", "quality_score")
if _quality_result is None:
    print("\nNot enough paired data for a bootstrap comparison.")
else:
    _q_point, _q_lo, _q_hi = _quality_result
    _q_crosses_zero = _q_lo <= 0 <= _q_hi
    if _q_crosses_zero:
        _q_verdict = "NOT a real improvement - could be noise. The rewrite step doesn't clearly help."
    elif _q_point > 0:
        _q_verdict = "a REAL, measurable quality improvement over the original docstrings."
    else:
        _q_verdict = "actually WORSE than the originals, a real decline - do not proceed."
    print(f"\nPaired bootstrap (rewritten minus original), 95% CI over 5,000 resamples: "
          f"{_q_point:+.3f}  (95% CI [{_q_lo:+.3f}, {_q_hi:+.3f}])")
    print(f"-> {_q_verdict}")


Rewriting and scoring 150 rows (of 400 in lora_train_df)...
  ...30/150 rewritten and scored
  ...60/150 rewritten and scored
  ...90/150 rewritten and scored
  ...120/150 rewritten and scored
  ...150/150 rewritten and scored

Done. 150/150 rows successfully rewritten.

Mean code-grounded quality score, original vs. rewritten:
run
original     3.053
rewritten    4.860
Name: quality_score, dtype: float64

Paired bootstrap (rewritten minus original), 95% CI over 5,000 resamples: +1.807  (95% CI [+1.633, +1.980])
-> a REAL, measurable quality improvement over the original docstrings.


## Step B: groundedness spot-check - does the rewrite invent anything?

A second LLM call per sampled row, asked specifically to look for claims in the
rewritten docstring that the code does not actually support. This is a manual-read aid
(the verdicts print for you to skim), not an automated pass/fail gate - a rewrite step
that quietly hallucinates plausible-sounding but wrong details would be worse than
useless as a training target, so this is worth reading before trusting Step A's
quality-score improvement at face value.

In [21]:
GROUNDEDNESS_PROMPT = (
    "Function code:\n```python\n{code}\n```\n\nDocstring to check:\n{doc}\n\n"
    "List any claims in the docstring that are NOT actually supported by the code "
    "(wrong parameter names, invented behavior, wrong return type, etc). If every claim "
    "is supported, reply with exactly: NONE. Otherwise, list each unsupported claim on "
    "its own line, briefly."
)


def groundedness_check(code_text, doc):
    try:
        resp = _client.chat.completions.create(
            model="gpt-4o-mini",
            messages=[{"role": "user", "content": GROUNDEDNESS_PROMPT.format(code=code_text, doc=doc)}],
            max_tokens=150, temperature=0,
        )
        return resp.choices[0].message.content.strip()
    except Exception as e:
        return f"call failed: {e}"


GROUNDEDNESS_CHECK_N = 12
GROUNDEDNESS_SEED = 77
_checked_idxs = pd.Series(list(rewritten_cache.keys())).sample(
    n=min(GROUNDEDNESS_CHECK_N, len(rewritten_cache)), random_state=GROUNDEDNESS_SEED
)

_n_clean = 0
for idx in _checked_idxs:
    row = lora_train_df.loc[idx]
    verdict = groundedness_check(row["func_code_no_doc"], rewritten_cache[idx])
    is_clean = verdict.strip().upper().startswith("NONE")
    _n_clean += int(is_clean)
    print(f"\n{'='*70}\nsample_idx={idx}  (repo: {row[repo_col]})\n{'='*70}")
    print(f"Original:  {str(row[doc_col])[:200]}")
    print(f"Rewritten: {rewritten_cache[idx][:300]}")
    print(f"Groundedness check: {verdict}")

print(f"\n{_n_clean}/{len(_checked_idxs)} sampled rewrites had no flagged unsupported claims. "
      f"Read the ones above that aren't clean before trusting the rewrite step.")



sample_idx=74  (repo: mikedh/trimesh)
Original:  The minimum volume primitive (box, sphere, or cylinder) that
        bounds the mesh.

        Returns
        ---------
        bounding_primitive : trimesh.primitives.Sphere
                        
Rewritten: """  
Determine the bounding primitive (oriented box, sphere, or cylinder) with the smallest volume that encloses the mesh. 

Returns
-------
bounding_primitive : trimesh.primitives.Box, trimesh.primitives.Sphere, or trimesh.primitives.Cylinder
    The primitive shape that bounds the mesh with the s
Groundedness check: - The docstring states "bounding_primitive : trimesh.primitives.Box, trimesh.primitives.Sphere, or trimesh.primitives.Cylinder", but the code does not specify the return type as being from the `trimesh.primitives` module. It only returns the bounding primitive with the smallest volume without confirming the specific class type.

sample_idx=299  (repo: SpamScope/mail-parser)
Original:  Exec msgconvert tool, to conv

## Gate: only proceed to training if Step A showed a real improvement

In [22]:
if _quality_result is None or _q_crosses_zero or _q_point <= 0:
    raise AssertionError(
        "Step A did not show a real quality improvement from the rewrite - training on "
        "these targets isn't justified by the evidence gathered so far. Stopping here "
        "rather than spending GPU time on an unproven change. Read Step B's groundedness "
        "output above for why, if the quality score itself looked fine but individual "
        "rewrites are unreliable."
    )
print("Step A showed a real quality improvement - proceeding to the pilot training runs.")


Step A showed a real quality improvement - proceeding to the pilot training runs.


## Step C: extend the rewrite to all 400 rows and train two pilot runs

Rewrites whatever's left of `lora_train_df` beyond Step A's 150-row sample (reusing
the cache, so nothing already rewritten is redone), then trains two LoRA runs at
EXACTLY the original `LoRA r=8` baseline's size and hyperparameters (400 rows, r=8,
2 epochs) - one on the untouched original targets, one on the rewritten targets - so
the only variable that differs between them is the training-target content itself.

In [23]:
_remaining_idxs = [i for i in lora_train_df.index if i not in rewritten_cache]
print(f"Rewriting the remaining {len(_remaining_idxs)} rows...")
for n, idx in enumerate(_remaining_idxs):
    row = lora_train_df.loc[idx]
    new_doc = rewrite_docstring(row["func_code_no_doc"], row[doc_col])
    if new_doc is not None:
        rewritten_cache[idx] = new_doc
    if (n + 1) % 50 == 0:
        print(f"  ...{n + 1}/{len(_remaining_idxs)} rewritten")

print(f"\nTotal rewritten: {len(rewritten_cache)}/{len(lora_train_df)}")

lora_train_df_rewritten = lora_train_df.copy()
lora_train_df_rewritten = lora_train_df_rewritten[lora_train_df_rewritten.index.isin(rewritten_cache.keys())].copy()
lora_train_df_rewritten[doc_col] = lora_train_df_rewritten.index.map(rewritten_cache)
print(f"Rewritten training set: {len(lora_train_df_rewritten)} rows "
      f"(dropped {len(lora_train_df) - len(lora_train_df_rewritten)} row(s) whose rewrite call failed).")

ORIGINAL_PILOT_LABEL = "LoRA r=8, original targets (pilot)"
REWRITTEN_PILOT_LABEL = "LoRA r=8, rewritten targets (pilot)"

original_pilot_dataset = SupervisedDocGenDataset(lora_train_df)
rewritten_pilot_dataset = SupervisedDocGenDataset(lora_train_df_rewritten)

training_runs = {}
_pilot_variant = {"label": ORIGINAL_PILOT_LABEL, "r": 8, "lora_alpha": 16}
training_runs[ORIGINAL_PILOT_LABEL] = train_lora_variant(
    _pilot_variant, original_pilot_dataset, loss_curve_eval_dataset, NB4_OUTPUT_DIR
)

_pilot_variant2 = {"label": REWRITTEN_PILOT_LABEL, "r": 8, "lora_alpha": 16}
training_runs[REWRITTEN_PILOT_LABEL] = train_lora_variant(
    _pilot_variant2, rewritten_pilot_dataset, loss_curve_eval_dataset, NB4_OUTPUT_DIR
)

print(f"\nTrained both pilot runs: {list(training_runs.keys())}")


Rewriting the remaining 250 rows...
  ...50/250 rewritten
  ...100/250 rewritten
  ...150/250 rewritten
  ...200/250 rewritten
  ...250/250 rewritten

Total rewritten: 400/400
Rewritten training set: 400 rows (dropped 0 row(s) whose rewrite call failed).

Training LoRA r=8, original targets (pilot) (lora_alpha=16, lr=0.00015, epochs=2, weight_decay=0.0, lora_dropout=0.05, use_best_checkpoint=False)


model.safetensors.index.json:   0%|          | 0.00/35.6k [00:00<?, ?B/s]

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/434 [00:00<?, ?it/s]

...

Saved adapter -> codesearchnet_lora_train_output/LoRA_r8,_original_targets_(pilot)

Training LoRA r=8, rewritten targets (pilot) (lora_alpha=16, lr=0.00015, epochs=2, weight_decay=0.0, lora_dropout=0.05, use_best_checkpoint=False)


Loading weights:   0%|          | 0/434 [00:00<?, ?it/s]

trainable params: 3,686,400 || all params: 3,089,625,088 || trainable%: 0.1193


Step,Training Loss,Validation Loss,Token Accuracy
10,0.858083,2.667539,0.483070
20,0.834749,2.611238,0.488337
30,0.792817,2.697825,0.477803
40,0.800015,2.779270,0.470278
50,0.761966,2.799419,0.471783


Saved adapter -> codesearchnet_lora_train_output/LoRA_r8,_rewritten_targets_(pilot)

Trained both pilot runs: ['LoRA r=8, original targets (pilot)', 'LoRA r=8, rewritten targets (pilot)']


## Step D: generate and score - Base, the original-target pilot, and the rewritten-target pilot

In [24]:
generation_results = []

print("Loading base model (no LoRA) for the baseline generation pass...")
base_model = AutoModelForCausalLM.from_pretrained(
    LORA_TARGET_MODEL_ID, dtype=torch.bfloat16 if device == "cuda" else torch.float32, device_map=device,
)
generation_results.append(run_generation_pass("Base (no LoRA)", base_model, tokenizer, eval_sample))
del base_model
gc.collect()
if device == "cuda":
    torch.cuda.empty_cache()
print("  Done with base model.")

for label, run in training_runs.items():
    print(f"Loading base model + {label} adapter for generation...")
    base_for_adapter = AutoModelForCausalLM.from_pretrained(
        LORA_TARGET_MODEL_ID, dtype=torch.bfloat16 if device == "cuda" else torch.float32, device_map=device,
    )
    gen_model = PeftModel.from_pretrained(base_for_adapter, run["adapter_path"])
    gen_model.eval()
    generation_results.append(run_generation_pass(label, gen_model, tokenizer, eval_sample))
    del gen_model, base_for_adapter
    gc.collect()
    if device == "cuda":
        torch.cuda.empty_cache()
    print(f"  Done with {label}.")

generation_df = pd.concat(generation_results, ignore_index=True)
print(f"\nTotal generations: {len(generation_df)} ({len(eval_sample)} rows x {generation_df['run'].nunique()} runs)")


Loading base model (no LoRA) for the baseline generation pass...


Loading weights:   0%|          | 0/434 [00:00<?, ?it/s]

  Done with base model.
Loading base model + LoRA r=8, original targets (pilot) adapter for generation...


Loading weights:   0%|          | 0/434 [00:00<?, ?it/s]

  Done with LoRA r=8, original targets (pilot).
Loading base model + LoRA r=8, rewritten targets (pilot) adapter for generation...


Loading weights:   0%|          | 0/434 [00:00<?, ?it/s]

  Done with LoRA r=8, rewritten targets (pilot).

Total generations: 210 (70 rows x 3 runs)


In [25]:
generation_df["rougeL"] = generation_df.apply(lambda r: rouge_l(r["true_doc"], r["generated_doc"]), axis=1)
generation_df["embed_cos_sim"] = generation_df.apply(lambda r: semantic_cos_sim(r["true_doc"], r["generated_doc"]), axis=1)
generation_df["judge_score"] = generation_df.apply(lambda r: judge_score(r["true_doc"], r["generated_doc"]), axis=1)

if _gemini_key:
    print(f"Judging {len(generation_df)} rows with Gemini ({len(generation_df)} API calls)...")
    generation_df["gemini_judge_score"] = generation_df.apply(lambda r: gemini_judge_score(r["true_doc"], r["generated_doc"]), axis=1)
else:
    generation_df["gemini_judge_score"] = np.nan

_metric_summary = generation_df.groupby("run")[["rougeL", "embed_cos_sim", "judge_score", "gemini_judge_score"]].mean().round(3)
print("\nMean metrics, all three runs:")
display(_metric_summary)


Judging 210 rows with Gemini (210 API calls)...

Mean metrics, all three runs:


,rougeL,embed_cos_sim,judge_score,gemini_judge_score
run,,,,
Base (no LoRA),0.222,0.594,3.129,3.900
"LoRA r=8, original targets (pilot)",0.293,0.615,2.743,3.414
"LoRA r=8, rewritten targets (pilot)",0.222,0.579,2.771,3.857


## Step E: is it real, and does rewriting the targets actually help?

Three paired comparisons per metric: each pilot run against Base (same test every
other run in this project gets), and - the one that actually isolates the rewrite's
effect - the rewritten-target pilot against the original-target pilot directly.

In [26]:
_metric_cols = [("rougeL", "ROUGE-L"), ("embed_cos_sim", "Semantic similarity"), ("judge_score", "GPT judge"),
                ("gemini_judge_score", "Gemini judge")]
_metric_cols = [(c, n) for c, n in _metric_cols if generation_df[c].notna().any()]


def _print_bootstrap(run_a, run_b, label_a, label_b):
    print(f"\nPaired bootstrap: {label_a} minus {label_b}, 95% CI over 5,000 resamples:")
    for col, nice_name in _metric_cols:
        result = bootstrap_paired_diff(generation_df, run_a, run_b, col)
        if result is None:
            print(f"  {nice_name}: not enough paired data.")
            continue
        point_est, ci_low, ci_high = result
        crosses_zero = ci_low <= 0 <= ci_high
        if crosses_zero:
            verdict = "NOT distinguishable - could be noise"
        elif point_est > 0:
            verdict = f"real, measurable IMPROVEMENT - {label_a} ahead"
        else:
            verdict = f"real, measurable DECLINE - {label_a} behind"
        print(f"  {nice_name}: {point_est:+.3f}  (95% CI [{ci_low:+.3f}, {ci_high:+.3f}])  -> {verdict}")


_print_bootstrap(ORIGINAL_PILOT_LABEL, "Base (no LoRA)", ORIGINAL_PILOT_LABEL, "Base")
_print_bootstrap(REWRITTEN_PILOT_LABEL, "Base (no LoRA)", REWRITTEN_PILOT_LABEL, "Base")
print(f"\n{'='*70}\nTHE KEY COMPARISON - isolates the rewrite's own effect:\n{'='*70}")
_print_bootstrap(REWRITTEN_PILOT_LABEL, ORIGINAL_PILOT_LABEL, REWRITTEN_PILOT_LABEL, ORIGINAL_PILOT_LABEL)



Paired bootstrap: LoRA r=8, original targets (pilot) minus Base, 95% CI over 5,000 resamples:
  ROUGE-L: +0.071  (95% CI [+0.042, +0.099])  -> real, measurable IMPROVEMENT - LoRA r=8, original targets (pilot) ahead
  Semantic similarity: +0.021  (95% CI [-0.008, +0.050])  -> NOT distinguishable - could be noise
  GPT judge: -0.386  (95% CI [-0.614, -0.186])  -> real, measurable DECLINE - LoRA r=8, original targets (pilot) behind
  Gemini judge: -0.486  (95% CI [-0.743, -0.229])  -> real, measurable DECLINE - LoRA r=8, original targets (pilot) behind

Paired bootstrap: LoRA r=8, rewritten targets (pilot) minus Base, 95% CI over 5,000 resamples:
  ROUGE-L: -0.001  (95% CI [-0.018, +0.016])  -> NOT distinguishable - could be noise
  Semantic similarity: -0.015  (95% CI [-0.037, +0.007])  -> NOT distinguishable - could be noise
  GPT judge: -0.357  (95% CI [-0.543, -0.171])  -> real, measurable DECLINE - LoRA r=8, rewritten targets (pilot) behind
  Gemini judge: -0.043  (95% CI [-0.271, +

## Conclusion — the rewrite is higher-quality in isolation, but doesn't clearly fix the LoRA gap

**Step A (quality of the rewrite itself):** a real, large improvement. The code-grounded rewrite
scored 4.860 vs. the original's 3.053 on the same judge prompt, a paired-bootstrap gain of +1.807
(95% CI [+1.633, +1.980]) - not noise, and large enough to justify moving on to the pilot training.

**Step B (groundedness):** worth reading before trusting Step A at face value. 5 of the 12
spot-checked rewrites had no flagged issues, but the other 7 had at least one claim the reviewing
model couldn't verify against the actual code (invented types, unstated defaults, assumptions
about behavior the code doesn't show). The rewrite is more detailed and better-organized, but
"more detailed" and "fully grounded in the code" are not the same thing, and this gate doesn't
block on that - it's a caution to weigh against Step E's results below.

**Step E (does it help the actual LoRA gap?): a genuinely mixed answer, not a clean win.**
The one comparison that isolates the rewrite's own effect - rewritten-target pilot minus
original-target pilot, same size/hyperparameters, same eval sample - split by metric:
- ROUGE-L: **real decline** (-0.071, 95% CI [-0.106, -0.037]) - the rewritten-target model is
  farther from the literal reference text than the original-target model was.
- Semantic similarity: **real decline** (-0.036, 95% CI [-0.067, -0.005]).
- GPT judge: **not distinguishable from noise** (+0.029, 95% CI [-0.186, +0.243]).
- Gemini judge: **real improvement** (+0.443, 95% CI [+0.186, +0.700]).

Two metrics get worse, one is a wash, and only one - Gemini - shows a real gain. Against base,
neither pilot clears the bar either: the original-target pilot repeats NB4's familiar pattern
(real ROUGE-L gain, real GPT- and Gemini-judge decline), and the rewritten-target pilot mostly
just erases the ROUGE-L/semantic-similarity gain (both now indistinguishable from base) while
still showing a real GPT-judge decline (-0.357) - Gemini is the only judge that stops declining
relative to base (-0.043, not distinguishable from noise).

**Taken together, this does not change NB4's final decision.** Rewriting the targets measurably
improves standalone docstring quality and moves one judge's verdict, but it does not produce a
run that clears "real improvement with no offsetting real decline" - the GPT-judge decline
persists, and ROUGE-L/semantic similarity go from real gains to noise. `FINAL_LORA_CHOICE =
"neither - base model retained"` still stands. The more interesting finding here is the
GPT/Gemini split itself: NB4 Step 6c already found the two judges don't fully agree on the
hardest rows, and this pilot adds another data point in the same direction - whether the
decline is a genuine content-quality gap or partly judge-calibration-sensitive is still not
settled, and a full-scale (not pilot) rewritten-target run with a larger eval sample would be
the natural next step if this project continues, ideally alongside a look at *which* rewrites
Step B flagged as ungrounded to see if excluding them changes the picture.
